# 03 – LLM Agents in Hotelling Competition

This notebook demonstrates how to run a simulation where one firm is controlled
by an LLM agent and the other by a naive agent.

> **Prerequisites**: set `OPENAI_API_KEY` in your environment, or configure a
> local Ollama endpoint via `base_url`.

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
from hotelling.agents.llm_agent import LLMAgent
from hotelling.agents.naive_agent import NaiveAgent
from hotelling.simulation.config import SimulationConfig
from hotelling.simulation.engine import SimulationEngine
from hotelling.simulation.metrics import aggregate_results

# Configure the LLM agent
# For local Ollama: base_url='http://localhost:11434/v1', api_key='ollama'
api_key = os.environ.get('OPENAI_API_KEY')  # or set explicitly

llm_firm = LLMAgent(
    firm_id='LLM_Firm',
    location=(0.3, 0.5),
    price=1.0,
    model='gpt-4o-mini',
    api_key=api_key,
    temperature=0.2,
)

naive_firm = NaiveAgent(
    firm_id='Naive_Firm',
    location=(0.7, 0.5),
    price=1.0,
    strategy='center',
    seed=42,
)

cfg = SimulationConfig(n_steps=30, n_consumers=1000, seed=42, log_every=5)
engine = SimulationEngine(config=cfg)
engine.setup([llm_firm, naive_firm])
results = engine.run()
agg = aggregate_results(results)

print(engine.market.summary())

In [ ]:
try:
    import matplotlib.pyplot as plt
    from hotelling.visualization.plots import plot_market_shares, plot_price_evolution

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_market_shares(agg, ax=axes[0])
    plot_price_evolution(agg, ax=axes[1])
    plt.suptitle('LLM vs Naive Agent', fontsize=14)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install matplotlib: pip install matplotlib')